<div class="alert alert-block alert-success">


# **03 — feature engineering pipeline**

## **Goal of this notebook**
The raw columns in the database (goals, minutes_played, date_of_birth, net_transfer_record) are not directly comparable or analytically useful on their own. This notebook derives meaningful metrics from the raw data so that every subsequent notebook has better inputs to work with. This is not analysis — it is preparation for analysis.

### **Some metrics to make**
Player-level metrics
- age — calculated from date_of_birth to a reference date (e.g. latest game date in dataset)
- contract_years_remaining — from contract_expiration_date to reference date
- goals_per_90 — goals / (minutes_played / 90), for players with sufficient appearances
- assists_per_90 — same approach
- goal_contributions_per_90 — (goals + assists) / (minutes_played / 90)
- minutes_played_ratio — minutes_played / (appearances × 90), a proxy for how often a player plays the full match vs gets substituted
- cards_per_90 — (yellow_cards + red_cards×2) / (minutes_played / 90)

Club-level metrics (aggregated from games and appearances)
- win_rate — wins / total_games
- goals_scored_per_game — total goals scored / total games
- goals_conceded_per_game — total goals conceded / total games
- goal_difference_per_game
- avg_attendance — average attendance across home games
- transfer_balance — using the cleaned numeric_net_transfer_record from notebook 02

Game-level metrics
- total_goals — home_club_goals + away_club_goals
- goal_difference — absolute difference between home and away goals
- is_draw, home_win, away_win — boolean outcome flags

##### **the changes are going to take place directly on the database not the raw data**

In [4]:
# importing the necessaries
import sys
import os
# Adding the root to the path to use utils folder
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.db_utils import run_query , execute_ddl
from utils.custom_plots import distribution_plot, outlier_plot
from utils.schema_diagram import schema_diagram


'''import plotly.io as pio
pio.renderers.default = "png" ''' # drop these 2 lines if you want interactive charts locally

'import plotly.io as pio\npio.renderers.default = "png" '

<div class="alert alert-block alert-success">

# First lets see the connections 
## here we can have a better understanding of cross-table metrics

In [9]:
database_diagram=schema_diagram(run_query_fn=run_query,
                                layout='circular',
                                edge_labels='always',
                                title='Football database schema',
                                edge_label_font_size=13
                                )

database_diagram

<div class="alert alert-block alert-success">

# Player-level metrics:


In [ ]:
# creating age column for players:


# we calculate the age based on the latest data which is 2025(not 2026) 
age_query = r'''
ALTER TABLE players
ADD COLUMN age INT GENERATED ALWAYS AS (
extract(year from age(cast('2025-01-01' as date), date_of_birth))
) STORED; 
'''

contract_remaining_date_query = r'''
ALTER TABLE players
ADD COLUMN contract_years_remaining NUMERIC(4, 1) GENERATED ALWAYS AS (
    ROUND(
        EXTRACT(EPOCH FROM AGE(contract_expiration_date, CAST('2025-01-01' AS DATE)))
        / (365.25 * 24 * 3600)
    , 1)
) STORED;
'''


# we write a query for metrics that need aggregation  
# GENERATED ALWAYS AS doesnt work for metrics that need computation on more then 1 row so instead we define columns and populate them with computed value
# we need to define a threshold for ratio denominato so it doesnt explode the ratio with low values 

aggregated_metrics_query = r'''
ALTER TABLE players
ADD COLUMN goals_per_90            NUMERIC(5, 2),
ADD COLUMN assists_per_90          NUMERIC(5, 2),
ADD COLUMN goal_contributions_per_90 NUMERIC(5, 2),
ADD COLUMN minutes_played_ratio    NUMERIC(4, 3),
ADD COLUMN cards_per_90            NUMERIC(5, 2),
ADD COLUMN total_appearances       INT,
ADD COLUMN total_minutes_played    INT;


UPDATE players p
SET
    total_appearances        = agg.appearances,
    total_minutes_played     = agg.total_minutes,
    goals_per_90             = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND(agg.total_goals / (agg.total_minutes / 90.0), 2) 
                               END,
    assists_per_90           = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND(agg.total_assists / (agg.total_minutes / 90.0), 2) 
                               END,
    goal_contributions_per_90 = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND((agg.total_goals + agg.total_assists) / (agg.total_minutes / 90.0), 2) 
                               END,
    minutes_played_ratio     = CASE 
                                 WHEN agg.appearances > 5
                                 THEN ROUND(agg.total_minutes / (agg.appearances * 90.0), 3) 
                               END,
    cards_per_90             = CASE 
                                 WHEN agg.total_minutes >= 200 
                                 THEN ROUND((agg.yellow_cards + agg.red_cards * 2) / (agg.total_minutes / 90.0), 2) 
                               END
FROM (
    SELECT
        player_id,
        COUNT(*)             AS appearances,
        SUM(minutes_played)  AS total_minutes,
        SUM(goals)           AS total_goals,
        SUM(assists)         AS total_assists,
        SUM(yellow_cards)    AS yellow_cards,
        SUM(red_cards)       AS red_cards
    FROM appearances
    GROUP BY player_id
) agg
WHERE p.player_id = agg.player_id;
'''
